# Runtime simulation for all explainers

This notebook benchmarks the explainers in `cxplain` on synthetic datasets with varying numbers of clusters, features, and observations. The goal is to approximate the runtime profile of the available explainers under controlled data sizes.

The grid includes:
- clusters: 5, 10, 20
- features: 5, 15, 30
- observations: 1000, 10000, 50000

For XKM, all flavours are benchmarked: next_best, all, within_scatter, scatter_ratio.

In [8]:
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from tqdm.auto import tqdm

from cxplain.neon import NeonKMeansExplainer
from cxplain.shap import ShapExplainer
from cxplain.tree import DecisionTreeExplainer, RandomForestExplainer
from cxplain.xkm import XkmExplainer

%matplotlib inline

In [14]:
@dataclass
class RuntimeResult:
    explainer: str
    n_clusters: int
    n_features: int
    n_obs: int
    run_idx: int
    runtime_sec: float
    status: str
    global_relevance_shape: Tuple[int, ...] | None = None

CLUSTER_COUNTS = [5, 10, 15]
FEATURE_COUNTS = [5, 10, 30]
OBSERVATION_COUNTS = [100, 1000, 10000]
N_RUNS_PER_EXPLAINER = 100
SEED = 42
XKM_FLAVOURS = ['next_best', 'all', 'within_scatter', 'scatter_ratio']
EXPLAINERS = [
    *[f'xkm_{flavour}' for flavour in XKM_FLAVOURS],
    'decision_tree',
    'random_forest',
    'shap',
    'neon',
]


def make_synthetic_dataset(n_clusters: int, n_features: int, n_obs: int, seed: int = SEED):
    X, labels, centers = make_blobs(
        n_samples=n_obs,
        n_features=n_features,
        centers=n_clusters,
        cluster_std=1.5,
        random_state=seed,
        return_centers=True,
    )
    return X, labels, centers


def benchmark_explainer(
    explainer_name: str,
    X: np.ndarray,
    labels: np.ndarray,
    cluster_centers: np.ndarray,
    run_idx: int,
):
    start = time.perf_counter()
    status = 'ok'
    relevance_shape = None

    try:
        if explainer_name.startswith('xkm_') or explainer_name == 'xkm':
            flavour = explainer_name.replace('xkm_', '', 1) if explainer_name.startswith('xkm_') else 'next_best'
            explainer = XkmExplainer(
                data=X,
                cluster_centers=cluster_centers,
                flavour=flavour,
                distance_metric='euclidean',
                cluster_predictions=labels,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        elif explainer_name == 'decision_tree':
            explainer = DecisionTreeExplainer(
                data=X,
                cluster_predictions=labels,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
                max_depth=5,
                random_state=SEED,
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        elif explainer_name == 'random_forest':
            explainer = RandomForestExplainer(
                data=X,
                cluster_predictions=labels,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
                n_estimators=25,
                max_depth=6,
                random_state=SEED,
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        elif explainer_name == 'shap':
            explainer = ShapExplainer(
                data=X,
                cluster_predictions=labels,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
                n_estimators=15,
                max_depth=5,
                random_state=SEED,
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        elif explainer_name == 'neon':
            explainer = NeonKMeansExplainer(
                data=X,
                cluster_centers=cluster_centers,
                predictions=labels,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        else:
            raise ValueError(f'Unknown explainer: {explainer_name}')

    except Exception as exc:
        status = f'error: {type(exc).__name__}: {exc}'

    elapsed = time.perf_counter() - start
    return RuntimeResult(
        explainer=explainer_name,
        n_clusters=cluster_centers.shape[0],
        n_features=X.shape[1],
        n_obs=X.shape[0],
        run_idx=run_idx,
        runtime_sec=elapsed,
        status=status,
        global_relevance_shape=relevance_shape,
    )


def run_runtime_simulation(
    cluster_counts: List[int] = CLUSTER_COUNTS,
    feature_counts: List[int] = FEATURE_COUNTS,
    observation_counts: List[int] = OBSERVATION_COUNTS,
    explainers: List[str] = EXPLAINERS,
    n_runs_per_explainer: int = N_RUNS_PER_EXPLAINER,
):
    runtime_records: List[RuntimeResult] = []

    total_steps = (
        len(cluster_counts)
        * len(feature_counts)
        * len(observation_counts)
        * len(explainers)
        * n_runs_per_explainer
    )

    with tqdm(total=total_steps, desc='Runtime simulation', unit='run') as pbar:
        for n_clusters in cluster_counts:
            for n_features in feature_counts:
                for n_obs in observation_counts:
                    X, labels, cluster_centers = make_synthetic_dataset(
                        n_clusters=n_clusters,
                        n_features=n_features,
                        n_obs=n_obs,
                    )

                    for explainer_name in explainers:
                        for run_idx in range(1, n_runs_per_explainer + 1):
                            runtime_records.append(
                                benchmark_explainer(
                                    explainer_name=explainer_name,
                                    X=X,
                                    labels=labels,
                                    cluster_centers=cluster_centers,
                                    run_idx=run_idx,
                                )
                            )
                            pbar.update(1)

    results_df = pd.DataFrame([record.__dict__ for record in runtime_records])
    results_df = results_df.sort_values(
        ['n_clusters', 'n_features', 'n_obs', 'explainer', 'run_idx']
    ).reset_index(drop=True)

    # Scenario-level mean runtime per explainer (using successful runs)
    summary_df = (
        results_df.assign(runtime_sec_ok=results_df['runtime_sec'].where(results_df['status'] == 'ok'))
        .groupby(['n_clusters', 'n_features', 'n_obs', 'explainer'], as_index=False)
        .agg(
            mean_runtime_sec=('runtime_sec_ok', 'mean'),
            n_runs=('run_idx', 'count'),
            n_success=('status', lambda s: (s == 'ok').sum()),
            n_errors=('status', lambda s: (s != 'ok').sum()),
        )
        .sort_values(['n_clusters', 'n_features', 'n_obs', 'explainer'])
        .reset_index(drop=True)
    )

    return results_df, summary_df

In [15]:
# Full benchmark: each explainer is executed 100 times per scenario.
results_df, summary_df = run_runtime_simulation(n_runs_per_explainer=1)

summary_df.head(20)

Runtime simulation: 100%|██| 216/216 [01:17<00:00,  2.80run/s]


,n_clusters,n_features,n_obs,explainer,mean_runtime_sec,n_runs,n_success,n_errors
0,5,5,100,decision_tree,0.004763,1,1,0
1,5,5,100,neon,0.034857,1,1,0
2,5,5,100,random_forest,0.106199,1,1,0
3,5,5,100,shap,0.069800,1,1,0
4,5,5,100,xkm_all,0.007548,1,1,0
5,5,5,100,xkm_next_best,0.013083,1,1,0
6,5,5,100,xkm_scatter_ratio,0.008949,1,1,0
7,5,5,100,xkm_within_scatter,0.005998,1,1,0
8,5,5,1000,decision_tree,0.010363,1,1,0
9,5,5,1000,neon,0.293031,1,1,0


In [ ]:
results_df.to_csv('runtime_simulation_results_raw.csv', index=False)
summary_df.to_csv('runtime_simulation_results_summary.csv', index=False)